# MTPL frequency: data ingestion

Read the source data, apply the production-intended transforms, and save the only governed model-frame handoff used by training.

In [ ]:
DATABASE_MODE = "local"  # "local" or "remote"
RUNTIME_MODULE = None  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False
DATA_AS_OF = "2026-06-30"  # Required dataset version stamp.
REFRESH_LOCAL_RAW = False
REPLACE_MODEL_FRAME = False

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sqlalchemy import text  # noqa: E402

from pricing_pipeline.data.fremtpl import load_fremtpl_raw  # noqa: E402
from pricing_pipeline.infra.schema import schema_names_from_connectable  # noqa: E402
from pricing_pipeline.notebook import connect, save_model_frame  # noqa: E402

MODEL_DIR = PROJECT_ROOT / "pricing_models/mtpl_frequency"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"

## Connect and verify the source/audit destination

In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

## Read source data from SQL

Local mode downloads the OpenML source only when needed. Remote mode never contacts OpenML; replace the query cell when the work source database differs from the audit destination.

In [ ]:
if pricing.mode == "local":
    local_source_rows = load_fremtpl_raw(pricing.engine, replace=REFRESH_LOCAL_RAW)
    display({"Local source rows": local_source_rows})

schemas = schema_names_from_connectable(pricing.engine)
SOURCE_SQL = f"""
SELECT
    IDpol, ClaimNb, Exposure, Area, VehPower, VehAge, DrivAge,
    BonusMalus, VehBrand, VehGas, Density, Region
FROM {schemas.pricing}.FREMTPL_RAW
ORDER BY IDpol
"""
raw = pd.read_sql_query(text(SOURCE_SQL), pricing.engine)
if raw.empty:
    raise RuntimeError("FREMTPL_RAW is empty; load source rows before modelling.")
display({"Rows loaded": len(raw), "Columns loaded": len(raw.columns)})

## Build the final model frame

In [ ]:
FEATURE_COLUMNS = (
    "VehAge", "DrivAge", "BonusMalus", "LogDensity", "Area",
    "VehPower", "VehBrand", "VehGas", "Region",
)
frame = raw.loc[raw["Exposure"].astype(float) > 0].copy()
frame["LogDensity"] = np.log(frame["Density"].astype(float).clip(lower=1.0))
frame["LogExposure"] = np.log(frame["Exposure"].astype(float))
frame["data_as_of"] = DATA_AS_OF
frame = (
    frame.loc[:, [
        "IDpol", "ClaimNb", "Exposure", "LogExposure", "data_as_of",
        *FEATURE_COLUMNS,
    ]]
    .sort_values("IDpol")
    .reset_index(drop=True)
)
display({"Rows modelled": len(frame), "Columns modelled": len(frame.columns)})

## Save the verified training handoff

In [ ]:
frame_artifact = save_model_frame(
    frame,
    FRAME_ARTIFACT_PATH,
    replace=REPLACE_MODEL_FRAME,
)
display(frame_artifact)